In [ ]:
import os
import warnings
import torch
import numpy as np
import matplotlib.pyplot as plt
from sympy import Symbol, Eq, Abs

In [ ]:
# importing required libraries
import modulus.sym

# NS library
from modulus.sym.eq.pdes.navier_stokes import NavierStokes

# Set up Geometry
from modulus.sym.geometry.primitives_2d import Rectangle

# library to set up the domain
from modulus.sym.domain import Domain

# Library to set BC
from modulus.sym.domain.constraint import (
    PointwiseBoundaryConstraint,
    PointwiseInteriorConstraint,
)

# Library to set up solver configuration
from modulus.sym.hydra import to_yaml
from modulus.sym.hydra.utils import compose
from modulus.sym.hydra.config import ModulusConfig

#from modulus.sym.hydra import to_absolute_path, instantiate_arch, ModulusConfig

# Library to create solver
from modulus.sym.solver import Solver

from modulus.sym.domain.validator import PointwiseValidator
from modulus.sym.domain.inferencer import PointwiseInferencer
from modulus.sym.key import Key
from modulus.sym.utils.io import (
    csv_to_dict,
    ValidatorPlotter,
    InferencerPlotter,
)

# Navier Stokes Equation

In [ ]:
ns = NavierStokes(nu=0.01, rho=1.0, dim=2, time=False)

In [ ]:
ns.equations

In [ ]:
ns.pprint()

In [ ]:
ns.equations['continuity']

# Neural Network


In [ ]:
from modulus.sym.models.fully_connected import FullyConnectedArch
from modulus.sym.models.activation import Activation


flow_net = FullyConnectedArch(
    input_keys=[Key("x"), Key("y")],
    output_keys=[Key("u"), Key("v"), Key("p")],
    layer_size=512,

)

In [ ]:
flow_net

In [ ]:
nodes = ns.make_nodes() + [flow_net.make_node(name="flow_network")]

In [ ]:
nodes

# Geometry

In [ ]:
height = 0.1
width = .1
x, y = Symbol("x"), Symbol("y")
rec = Rectangle((-width/2, -height/2), (width/2, height/2))

# Constraints

## Domain

In [ ]:
ldc_domain = Domain() # Domain contains PDE and Boundary condition

## Boundary Condition

In [ ]:
# top wall
top_wall = PointwiseBoundaryConstraint(
    nodes = nodes,
    geometry = rec,
    outvar = {"u" : 1, "v" : 0},
    batch_size = 1000,
    lambda_weighting = {"u":
                      1.0 - 20 * Abs(x)
                      , "v": 1.0},        # weighting is does to avoid discontinuties
    criteria = Eq(y, height/2)
)

ldc_domain.add_constraint(top_wall, "top_wall")

# no_slip
no_slip = PointwiseBoundaryConstraint(
    nodes = nodes,
    geometry = rec,
    outvar = {"u" : 0, "v" : 0},
    batch_size = 1000,
    criteria = y<height/2
)


ldc_domain.add_constraint(no_slip, "no_slip")

## Interior Constraints

In [ ]:
# Interior
interior = PointwiseInteriorConstraint(
    nodes=nodes,
    geometry=rec,
    outvar={"continuity": 0, "momentum_x": 0, "momentum_y": 0},
    batch_size=4000,
    lambda_weighting={
        # Signed Distance Field(SDF) :
        # points away from the boundary have a larger weight compared to the ones closer to the boundary.
        # faster convergence

        "continuity": Symbol("sdf"),
        "momentum_x": Symbol("sdf"),
        "momentum_y": Symbol("sdf"),
    }
)

ldc_domain.add_constraint(interior, "interior")

# Solver Configuration

In [ ]:
%%writefile config.yaml

defaults:
  - modulus_default
  - arch: fully_connected
  - scheduler: tf_exponential_lr
  - optimizer: adam
  - loss: sum
  - _self_

scheduler:
  decay_rate: 0.95
  decay_steps: 4000

training:
  rec_validation_freq: 1000
  rec_inference_freq: 2000
  rec_monitor_freq: 1000
  rec_constraint_freq: 2000
  max_steps: 10000

batch_size:
  TopWall: 1000
  NoSlip: 1000
  Interior: 4000

graph:
  func_arch: true


In [ ]:
cfg = compose(config_path=".", config_name="config")
cfg.network_dir = 'outputs' # Set the network directory for checkpoints
print(to_yaml(cfg))

# Create Solver

In [ ]:
slv = Solver(cfg, ldc_domain)

# Run Solver

In [ ]:
import logging

# Create a function to configure logging
def configure_logging():
    # Get the root logger
    logger = logging.getLogger()
    logger.setLevel(logging.DEBUG)

    # Remove all existing handlers
    while logger.handlers:
        logger.handlers.pop()

    # Create a stream handler
    stream_handler = logging.StreamHandler()
    stream_handler.setLevel(logging.DEBUG)

    # Create a formatter
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    stream_handler.setFormatter(formatter)

    # Add the handler to the logger
    logger.addHandler(stream_handler)

# Configure logging
configure_logging()

# Test logging
logging.debug("Logging is configured correctly.")

In [ ]:
slv.solve()

In [ ]:
# help(interior)

# Output Files

## Load Model from File

In [ ]:
torch.load('./outputs/optim_checkpoint.0.pth').keys()

In [ ]:
torch.load('./outputs/flow_network.0.pth').keys()

In [ ]:
checkpoint = torch.load('./outputs/flow_network.0.pth')
try:
    checkpoint.eval()
except AttributeError as error:
    print(error)

flow_net.load_state_dict(checkpoint)

## Evaluate PyTorch model

In [ ]:
flow_net

In [ ]:
torch.randn(5, 1)

In [ ]:
torch.tensor([0.04])


In [ ]:
torch.rand(5, 1)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# a top wall point (x,y) = [0, .05]
flow_net({"x": torch.tensor([0],device=device), "y": torch.tensor([0.05],device=device)} )

In [ ]:
(torch.rand(5, 1) * 0.1) - 0.05

In [ ]:
flow_net({"x": ((torch.rand(5, 1) * 0.1) -0.05).to(device), "y": ((torch.rand(5, 1) * 0.1) -0.05).to(device)} )

In [ ]:
flow_net({"x": torch.tensor([0.05],device=device), "y": torch.tensor([0],device=device)} )

# Visulization of results

In [ ]:
# Define the domain
x_values = np.linspace(-0.05, 0.05, 100)
y_values = np.linspace(-0.05, 0.05, 100)
x_grid, y_grid = np.meshgrid(x_values, y_values)

# Flatten the grid arrays to create inputs for the model
x_flat = x_grid.flatten()
y_flat = y_grid.flatten()

# Convert to PyTorch tensors
x_tensor = torch.tensor(x_flat, dtype=torch.float32).unsqueeze(1).to(device)
y_tensor = torch.tensor(y_flat, dtype=torch.float32).unsqueeze(1).to(device)

# Create input dictionary for the model
inputs = {"x": x_tensor, "y": y_tensor}


In [ ]:
"""
x_values = np.linspace(-0.05, 0.05, 5)
y_values = np.linspace(-0.05, 0.05, 5)
x_grid, y_grid = np.meshgrid(x_values, y_values)
print(x_grid.shape)
print(x_values.shape)

print(x_values)
print(x_grid)

print(x_values.ndim)
print(x_grid.ndim)
"""

In [ ]:
outputs = flow_net(inputs)
u_flat = outputs['u'].cpu().detach().numpy() # extracting output "u" and moving it to cpu and detaching so pytorch donot track it and to numpy
v_flat = outputs['v'].cpu().detach().numpy()
p_flat = outputs['p'].cpu().detach().numpy()

# Reshape the outputs back to grid shape
u_grid = u_flat.reshape(x_grid.shape)
v_grid = v_flat.reshape(x_grid.shape)
p_grid = p_flat.reshape(x_grid.shape)

In [ ]:
# Plot the color maps
fig, axs = plt.subplots(1, 3, figsize=(18, 6))

# Plot p
im0 = axs[0].imshow(p_grid, extent=[-0.05, 0.05, -0.05, 0.05], origin='lower', aspect='auto')
axs[0].set_title('p')
axs[0].set_xlabel('x')
axs[0].set_ylabel('y')
fig.colorbar(im0, ax=axs[0])

# Plot u
im1 = axs[1].imshow(u_grid, extent=[-0.05, 0.05, -0.05, 0.05], origin='lower', aspect='auto')
axs[1].set_title('u')
axs[1].set_xlabel('x')
axs[1].set_ylabel('y')
fig.colorbar(im1, ax=axs[1])

# Plot v
im2 = axs[2].imshow(v_grid, extent=[-0.05, 0.05, -0.05, 0.05], origin='lower', aspect='auto')
axs[2].set_title('v')
axs[2].set_xlabel('x')
axs[2].set_ylabel('y')
fig.colorbar(im2, ax=axs[2])

plt.tight_layout()
plt.show()

In [ ]:
# Define the domain
x_values = np.linspace(-0.05, 0.05, 30)  # Reduced number of points for clearer vector field
y_values = np.linspace(-0.05, 0.05, 30)
x_grid, y_grid = np.meshgrid(x_values, y_values)

# Flatten the grid arrays to create inputs for the model
x_flat = x_grid.flatten()
y_flat = y_grid.flatten()

# Convert to PyTorch tensors
x_tensor = torch.tensor(x_flat, dtype=torch.float32).unsqueeze(1).to(device)
y_tensor = torch.tensor(y_flat, dtype=torch.float32).unsqueeze(1).to(device)

# Create input dictionary for the model
inputs = {"x": x_tensor, "y": y_tensor}

# Pass inputs through the model to get predictions
outputs = flow_net(inputs)
u_flat = outputs['u'].cpu().detach().numpy()
v_flat = outputs['v'].cpu().detach().numpy()

# Reshape the outputs back to grid shape
u_grid = u_flat.reshape(x_grid.shape)
v_grid = v_flat.reshape(x_grid.shape)

# Plot the vector field
plt.figure(figsize=(8, 6))
plt.quiver(x_grid, y_grid, u_grid, v_grid, scale=7, color='blue')  # Adjust scale as needed
plt.title('Vector Field of u and v')
plt.xlabel('x')
plt.ylabel('y')
plt.xlim([-0.05, 0.05])
plt.ylim([-0.05, 0.05])
plt.grid(True)
plt.show()

In [ ]:
# Calculate the magnitude of the vectors
magnitude = np.sqrt(u_grid**2 + v_grid**2)

# Plot the vector field with colors
plt.figure(figsize=(8, 6))
plt.quiver(x_grid, y_grid, u_grid, v_grid, magnitude, scale=10, cmap='viridis')  # Adjust scale as needed
plt.colorbar(label='Magnitude')
plt.title('Vector Field of u and v')
plt.xlabel('x')
plt.ylabel('y')
plt.xlim([-0.05, 0.05])
plt.ylim([-0.05, 0.05])
plt.grid(True)
plt.show()